---
layout: post
courses: { csa: {week: 3} }
categories: [Java, Anatomy-of-a-Class]
lesson_language: Java
lesson_topic: Anatomy-of-a-Class
lesson_part: interactive
lesson_type: lesson
toc: true
codemirror: true
title: Anatomy of a Class
menu: nav/csa_units/csaunit3.html
permalink: /csa/unit_03/3_3
---


<div class="info-box">
    <h3>Anatomy of a Class</h3>
    <p>Every class in this lesson is a simplified, runnable model of a real piece of Open Coding Society's own password-reset system. Same fields, same control flow, same reasons -- just small enough to read start to finish.</p>
</div>

<div class="info-box key-points">
    <h3>Objects Used</h3>
    <p>The <code>System</code> class calls static methods <code>System.out.print</code>/<code>println</code> to write output. The <code>Math</code> class calls static <code>Math.random()</code> to generate a reset token's random component.</p>
</div>

<div class="info-box key-points">
    <h3>Other College Board Topics</h3>
    <p>A <strong>2D array</strong> pairs every possible outcome of a reset request with the response it produces. A <strong>switch</strong> control structure routes an outcome to that response -- with several outcomes deliberately falling through to the same case.</p>
</div>

<div class="info-box challenge-box">
    <h3>Key PBL Topic</h3>
    <p>A <code>PasswordResetRequest</code> object created with <code>new PasswordResetRequest()</code> encapsulates one attempt to reset a password -- its own fields, its own method for deciding what response to send back. That's the same idea behind how a Spring Boot controller owns a web request: an object wraps a whole piece of behavior, not just data.</p>
</div>


## Anatomy of a Password-Reset Request

The real OAuth-verified reset endpoint checks several different things before it decides what to tell the user: is the uid real? does the sid match? is this an admin account? has the rate limit been hit? Every one of those checks can fail for a *different* reason internally -- but almost all of them have to look **identical** from the outside.

> **From the real pipeline:** "Every denial path returns an identical response body (`{"verified":false}`), regardless of which check failed -- this is deliberate, so the endpoint can't be used to enumerate valid uid/sid pairs." The one exception is the rate-limit response, which has to look different so the frontend can show "Request a Ticket Instead."

Run the demo below. It models exactly that: a `switch` where three completely different failure reasons all fall through to the same response, and only the rate-limit case gets its own.


In [ ]:
// CODE_RUNNER: Press Run and notice: three different denial reasons produce the exact same response. That's not a bug.
import java.lang.Math;

public class PasswordResetRequest {
    public static final int SUCCESS = 0;
    public static final int UNKNOWN_UID = 1;
    public static final int SID_MISMATCH = 2;
    public static final int ADMIN_ACCOUNT = 3;
    public static final int RATE_LIMITED = 4;

    // 2D array: every outcome paired with a human-readable label for this demo.
    public final String[][] OUTCOMES = {
        {"SUCCESS", "a real reset token was issued"},
        {"UNKNOWN_UID", "no account with that uid exists"},
        {"SID_MISMATCH", "the last 5 digits of sid didn't match"},
        {"ADMIN_ACCOUNT", "admin and seeded accounts can't self-reset"},
        {"RATE_LIMITED", "3 requests already used in this window"},
    };

    String uid;

    PasswordResetRequest(String uid) {
        this.uid = uid;
    }

    // Generates a fake token string -- just enough to show Math.random() at work.
    // The real system signs a token with HMAC-SHA256 and a 5-minute TTL.
    String generateToken() {
        int nonce = (int) (Math.random() * 900000) + 100000;
        return uid + "-" + nonce;
    }

    // SWITCH -- three unrelated failure reasons deliberately fall through to the
    // same response, so an attacker can't tell WHY a request was denied.
    String respond(int outcome) {
        switch (outcome) {
            case SUCCESS:
                return "200 verified:true token=" + generateToken();
            case RATE_LIMITED:
                return "429 -- try again later, or request a ticket";
            case UNKNOWN_UID:
            case SID_MISMATCH:
            case ADMIN_ACCOUNT:
                return "403 verified:false";
            default:
                return "unexpected outcome";
        }
    }

    public static void main(String[] args) {
        PasswordResetRequest req = new PasswordResetRequest("toby");
        int[] outcomesToTry = {
            PasswordResetRequest.SUCCESS,
            PasswordResetRequest.UNKNOWN_UID,
            PasswordResetRequest.SID_MISMATCH,
            PasswordResetRequest.ADMIN_ACCOUNT,
            PasswordResetRequest.RATE_LIMITED,
        };
        for (int outcome : outcomesToTry) {
            System.out.println(req.OUTCOMES[outcome][0] + " (" + req.OUTCOMES[outcome][1] + ")");
            System.out.println("  -> " + req.respond(outcome));
        }
    }
}
PasswordResetRequest.main(null);


<div class="info-box">
    <h3>Popcorn Hack 1: Enumeration Resistance</h3>
    <p><code>UNKNOWN_UID</code>, <code>SID_MISMATCH</code>, and <code>ADMIN_ACCOUNT</code> all fall through to the exact same <code>return</code> in <code>respond()</code>. Why does the real system do this on purpose, instead of returning a more specific error for each one?</p>
    <p>
        <button class="btn small-btn" onclick="u33CheckPH1(0)">Specific errors would be slower to generate</button><br>
        <button class="btn small-btn" onclick="u33CheckPH1(1)">A specific error for "unknown uid" vs. "sid mismatch" would let an attacker figure out which uids are real accounts</button><br>
        <button class="btn small-btn" onclick="u33CheckPH1(2)">Java requires every switch case in a group to return the same thing</button><br>
        <button class="btn small-btn" onclick="u33CheckPH1(3)">It's a leftover from an older version of the code with no real reason</button>
    </p>
    <div id="u33-ph1-feedback"></div>
</div>

<script>
function u33CheckPH1(sel) {
    const fb = document.getElementById('u33-ph1-feedback');
    if (sel === 1) {
        fb.className = 'info-box success';
        fb.innerHTML = '<p><strong>Correct.</strong> If "unknown uid" and "sid mismatch" returned different responses, an attacker could try every possible uid and learn exactly which ones are real accounts -- without ever knowing a real sid. Collapsing them into one identical response closes that leak. Rate-limiting is the one deliberate exception, since the frontend genuinely needs to know when to show a different button.</p>';
    } else {
        fb.className = 'info-box warning-box';
        fb.innerHTML = '<p>Not quite -- this is a real, deliberate security decision documented in the actual pipeline. Think about what an attacker could learn if each denial reason looked different from the outside.</p>';
    }
}
</script>


## Hacks: Extend the Request

<div class="info-box">
    <h3>Documentation</h3>
    <p>Using Markdown cells and triple-backtick code fragments, describe your own work and answer:</p>
    <ul>
        <li>Where is a <strong>Class</strong> defined?</li>
        <li>Where is an <strong>instance</strong> of a Class created?</li>
        <li>Where is an object <strong>calling a method</strong>?</li>
        <li>Where is an object <strong>mutating data</strong>?</li>
        <li>Which outcomes fall through to the same <code>switch</code> case, and why?</li>
    </ul>
</div>

<div class="info-box">
    <h3>Build It</h3>
    <p>Add a <strong>6th outcome</strong> to <code>PasswordResetRequest</code>: <code>TICKET_GRANTED</code> -- the real escape hatch when someone hits the rate limit and an admin manually approves a ticket. Add it to <code>OUTCOMES</code>, give it its own case in <code>respond()</code>, and add it to the loop in <code>main</code>.</p>
    <ul>
        <li>Use <strong>constructors and instance data</strong>. <code>req</code> is activated with <code>new</code> -- make sure you understand what that keyword actually does.</li>
        <li>Use <strong>static methods and data</strong>. The <a href="https://www.javatpoint.com/java-math">Math class</a> performs its calculations through static methods -- yours can too.</li>
    </ul>
</div>


## The Object Superclass

<div class="info-box">
    <h3>Learning Targets</h3>
    <ul>
        <li>What is the <code>Object</code> class? Every class you write inherits from it, whether you write <code>extends</code> or not.</li>
        <li>Why does it matter? You can't opt out of it -- and overriding one of its methods comes with rules you must follow.</li>
    </ul>
</div>

Every class you write **without** an explicit `extends` is implicitly extended from the <a href="https://docs.oracle.com/javase/8/docs/api/java/lang/Object.html"><code>Object</code></a> superclass -- so it inherits a handful of methods for free, including `getClass()`, `toString()`, and `equals()`.

That matters the moment you try to override one: since every `Object` method is `public`, your override **must** also be `public` -- Java will not let you *narrow* the access on an inherited method. (`getClass()` is the one exception on this list -- it's `final`, so you can't override it at all.)

A login token is just an object like any other. Here's a `Token` class -- constructor, instance fields, and (broken, at first) a `toString()` override. Run the broken version below and read the actual compiler error Java gives you -- it names the exact rule you just broke.


In [ ]:
// CODE_RUNNER: Press Run and read the error message javac produces.
public class Token {
    String uid;
    int tokenVersion;

    Token(String uid, int tokenVersion) {
        this.uid = uid;
        this.tokenVersion = tokenVersion;
    }

    // this narrows Object's public toString() down to package-private -- illegal
    String toString() {
        return uid + " (tokenVersion " + tokenVersion + ")";
    }

    public static void main(String[] args) {
        Token t = new Token("toby", 3);
        System.out.println(t.toString());
    }
}
Token.main(null);


Now the fixed version -- `@Override` isn't what makes this legal (it's optional), but it *is* what makes the compiler double-check your work for you. Without a working `toString()` override, every line below would print something like `Token@1b6d3586` instead of who the token actually belongs to.


In [ ]:
// CODE_RUNNER: Press Run and compare the output of the explicit call vs. printing the object directly.
public class Token {
    String uid;
    int tokenVersion;

    Token(String uid, int tokenVersion) {
        this.uid = uid;
        this.tokenVersion = tokenVersion;
    }

    @Override
    public String toString() {
        return uid + " (tokenVersion " + tokenVersion + ")";
    }

    public static void main(String[] args) {
        Token t = new Token("toby", 3);
        System.out.println(t.toString()); // explicit call
        System.out.println(t);            // println calls toString() for you
    }
}
Token.main(null);


<div class="info-box">
    <h3>Popcorn Hack 2: Reading the Error</h3>
    <p>Why does the first <code>Token</code> above fail to compile?</p>
    <p>
        <button class="btn small-btn" onclick="u33CheckPH2(0)">It's missing the <code>@Override</code> annotation, which Java requires</button><br>
        <button class="btn small-btn" onclick="u33CheckPH2(1)">It returns a weaker access level (package-private) than <code>Object</code>'s public <code>toString()</code></button><br>
        <button class="btn small-btn" onclick="u33CheckPH2(2)"><code>toString()</code> cannot be overridden at all</button><br>
        <button class="btn small-btn" onclick="u33CheckPH2(3)">Token needs a no-argument constructor first</button>
    </p>
    <div id="u33-ph2-feedback"></div>
</div>

<script>
function u33CheckPH2(sel) {
    const fb = document.getElementById('u33-ph2-feedback');
    if (sel === 1) {
        fb.className = 'info-box success';
        fb.innerHTML = '<p><strong>Correct.</strong> Every method inherited from <code>Object</code> is <code>public</code>. Leaving off the modifier makes the override package-private, which weakens access -- javac refuses to compile it. <code>@Override</code> is optional (it just asks the compiler to verify you\'re actually overriding something); it isn\'t what caused this specific failure.</p>';
    } else {
        const msgs = [
            '<strong>Not quite.</strong> <code>@Override</code> is optional -- it tells the compiler to double-check you\'re actually overriding a real method, but leaving it off is not itself an error.',
            '',
            '<strong>Not quite.</strong> <code>toString()</code> is one of the most commonly overridden <code>Object</code> methods (<code>getClass()</code> is the one that\'s <code>final</code> and truly can\'t be overridden).',
            '<strong>Not quite.</strong> <code>Token</code> already has a constructor -- that\'s not the problem here.'
        ];
        fb.className = 'info-box warning-box';
        fb.innerHTML = '<p>' + msgs[sel] + '</p>';
    }
}
</script>


## Introduction to Inheritance

Inheritance lets one class (the **subclass**) reuse the fields and methods of another (the **superclass**), instead of retyping them.

<div class="info-box key-points">
    <h3>Key Concepts</h3>
    <ul>
        <li><strong>Superclass (parent)</strong> -- defines the shared attributes and behavior.</li>
        <li><strong>Subclass (child)</strong> -- declared with <code>extends</code>; inherits the superclass, and can add or override its own members.</li>
    </ul>
</div>

The real system tracks every login the same base way, no matter where it came from -- a browser tab or a mobile app. `Session` below defines that shared behavior. `WebSession` and `MobileSession` both `extend Session` -- run it and see what each subclass gets without writing a line of that logic itself.


In [ ]:
// CODE_RUNNER: Press Run and check: does mobileSession have active/invalidate() even though MobileSession never declares them?
public class SessionDemo {
    public static void main(String[] args) {
        WebSession webSession = new WebSession();
        System.out.println("webSession.active (inherited from Session): " + webSession.active);
        webSession.invalidate();
        System.out.println("webSession.active after invalidate(): " + webSession.active);

        MobileSession mobileSession = new MobileSession();
        System.out.println("mobileSession.active (inherited from Session): " + mobileSession.active);
        System.out.println("mobileSession.deviceId (Mobile's own field): " + mobileSession.deviceId);
    }
}

class Session {
    boolean active = true;
    void invalidate() {
        this.active = false;
    }
}

class WebSession extends Session {
    String cookieName = "session_token";
}

class MobileSession extends Session {
    String deviceId = "device-A1";
}
SessionDemo.main(null);


<div class="info-box">
    <h3>Popcorn Hack 3: The Keyword</h3>
    <p>Which keyword lets <code>WebSession</code> inherit fields and methods from <code>Session</code>?</p>
    <p>
        <button class="btn small-btn" onclick="u33CheckPH3(0)"><code>implements</code></button><br>
        <button class="btn small-btn" onclick="u33CheckPH3(1)"><code>extends</code></button><br>
        <button class="btn small-btn" onclick="u33CheckPH3(2)"><code>inherits</code></button><br>
        <button class="btn small-btn" onclick="u33CheckPH3(3)"><code>super</code></button>
    </p>
    <div id="u33-ph3-feedback"></div>
</div>

<script>
function u33CheckPH3(sel) {
    const fb = document.getElementById('u33-ph3-feedback');
    if (sel === 1) {
        fb.className = 'info-box success';
        fb.innerHTML = '<p><strong>Correct.</strong> <code>class WebSession extends Session</code> makes <code>Session</code> the superclass and <code>WebSession</code> the subclass. <code>super</code> is related but different -- it\'s used <em>inside</em> a subclass to reach the superclass\'s constructor or members, not to declare the relationship itself.</p>';
    } else {
        const msgs = [
            '<strong>Not quite.</strong> <code>implements</code> is for interfaces, not for inheriting from another class.',
            '',
            '<strong>Not quite.</strong> Java has no <code>inherits</code> keyword.',
            '<strong>Not quite.</strong> <code>super</code> is used inside a subclass to reach the superclass\'s constructor/members -- it doesn\'t declare the inheritance relationship itself.'
        ];
        fb.className = 'info-box warning-box';
        fb.innerHTML = '<p>' + msgs[sel] + '</p>';
    }
}
</script>


## Quick Check: Anatomy of a Class

<div class="info-box">
    <h3>5 Questions -- Password Resets, Object, and Inheritance</h3>
    <p>See how much stuck before moving to the next lesson</p>
</div>

<div id="u33-quiz-container"></div>
<div id="u33-quiz-score"></div>

<script>
(function() {
    const U33_QUIZ = [
        {
            q: 'In <code>PasswordResetRequest</code>, what is <code>OUTCOMES</code>?',
            choices: [
                'A method that generates tokens',
                'A 2D array pairing each outcome name with a human-readable label',
                'A static method on <code>System</code>',
                'A subclass of <code>PasswordResetRequest</code>'
            ],
            correct: 1,
            explain: '<code>OUTCOMES</code> is declared as <code>String[][]</code>, with each row holding an outcome name and a label describing it.'
        },
        {
            q: 'Why do <code>UNKNOWN_UID</code>, <code>SID_MISMATCH</code>, and <code>ADMIN_ACCOUNT</code> all fall through to the same <code>switch</code> case in <code>respond()</code>?',
            choices: [
                'So an attacker can\'t tell which failure reason actually happened, and can\'t use the response to figure out which uids are real',
                'Because Java requires grouped case labels to return the same value',
                'It\'s just to keep the method shorter',
                'Because those three outcomes are actually the same thing internally'
            ],
            correct: 0,
            explain: 'This is enumeration resistance: if each denial reason produced a different response, an attacker could probe uids one at a time and learn which ones are real accounts. Collapsing them into one identical response closes that leak.'
        },
        {
            q: 'Which of these is <strong>not</strong> a method every class inherits from <code>Object</code>?',
            choices: [
                '<code>getClass()</code>',
                '<code>toString()</code>',
                '<code>equals()</code>',
                '<code>invalidate()</code>'
            ],
            correct: 3,
            explain: '<code>invalidate()</code> is defined on <code>Session</code> in this lesson\'s inheritance example -- it has nothing to do with <code>Object</code>. The other three are real <code>Object</code> methods every class gets automatically.'
        },
        {
            q: 'Why must an override of <code>toString()</code> be declared <code>public</code>?',
            choices: [
                'Because <code>String</code> methods are always public',
                'Because Java will not let a subclass narrow the access level of an inherited method',
                'Because <code>@Override</code> requires it',
                'It doesn\'t have to be -- it\'s just a style convention'
            ],
            correct: 1,
            explain: 'Every <code>Object</code> method is <code>public</code>. An override can widen access but never narrow it, so dropping the <code>public</code> modifier is a compile error, not a style choice.'
        },
        {
            q: 'Given <code>class WebSession extends Session</code>, which statement is true?',
            choices: [
                '<code>Session</code> is the subclass and <code>WebSession</code> is the superclass',
                '<code>WebSession</code> inherits <code>Session</code>\'s fields and methods and can add its own',
                '<code>WebSession</code> and <code>Session</code> share no relationship at compile time',
                '<code>WebSession</code> must redeclare every field <code>Session</code> has'
            ],
            correct: 1,
            explain: '<code>WebSession</code> is the subclass, <code>Session</code> is the superclass. <code>WebSession</code> automatically has everything <code>Session</code> declares (like <code>active</code> and <code>invalidate()</code>) plus whatever it adds itself (like <code>cookieName</code>) -- no redeclaration needed.'
        }
    ];

    let u33Score = 0;
    let u33Answered = 0;
    const u33Locked = new Array(U33_QUIZ.length).fill(false);

    function u33RenderQuiz() {
        const container = document.getElementById('u33-quiz-container');
        container.innerHTML = '';
        U33_QUIZ.forEach((q, qi) => {
            let html = '<div class="info-box">';
            html += '<p><strong>Question ' + (qi + 1) + ' of ' + U33_QUIZ.length + ':</strong> ' + q.q + '</p>';
            html += '<p>';
            q.choices.forEach((c, ci) => {
                html += '<button class="btn small-btn" onclick="u33QuizAnswer(' + qi + ',' + ci + ')">' + c + '</button><br>';
            });
            html += '</p>';
            html += '<div id="u33-q' + qi + '-feedback"></div>';
            html += '</div>';
            container.innerHTML += html;
        });
    }

    window.u33QuizAnswer = function(qi, ci) {
        if (u33Locked[qi]) return;
        u33Locked[qi] = true;

        const q = U33_QUIZ[qi];
        const feedback = document.getElementById('u33-q' + qi + '-feedback');

        if (ci === q.correct) {
            feedback.className = 'info-box success';
            feedback.innerHTML = '<p><strong>Correct.</strong> ' + q.explain + '</p>';
            u33Score++;
        } else {
            feedback.className = 'info-box warning-box';
            feedback.innerHTML = '<p><strong>Incorrect.</strong> ' + q.explain + '</p>';
        }

        u33Answered++;
        if (u33Answered >= U33_QUIZ.length) {
            const scoreDiv = document.getElementById('u33-quiz-score');
            const pct = Math.round((u33Score / U33_QUIZ.length) * 100);
            scoreDiv.className = 'info-box' + (u33Score === U33_QUIZ.length ? ' success' : '');
            scoreDiv.innerHTML = '<p><strong>Your Score: ' + u33Score + '/' + U33_QUIZ.length + '</strong></p>' +
                '<div class="progress-bar"><div class="progress-fill" style="width:' + pct + '%"></div></div>';
        }
    };

    if (document.readyState === 'loading') document.addEventListener('DOMContentLoaded', u33RenderQuiz);
    else u33RenderQuiz();
})();
</script>


## Topics to Explore

Keep exploring the navigation bar's other units to see more of the CollegeBoard curriculum -- constructors (3.4) and access control (3.5/3.6) build directly on everything above.
